In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("../../Data/Monday_Combined_dataset.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

# First differences
df["d_gas_austin"] = df["gas_price"].diff()
df["d_gas_us"] = df["national_gas_price"].diff()
df["d_wti"] = df["wti_price"].diff()

df = df.dropna().copy()

In [11]:
def run_adl(df, y_col, x_col, p, q):
    temp = df.copy()

    for i in range(1, p + 1):
        temp[f"{y_col}_lag{i}"] = temp[y_col].shift(i)

    for i in range(0, q + 1):
        temp[f"{x_col}_lag{i}"] = temp[x_col].shift(i)

    temp = temp.dropna()

    y = temp[y_col]

    X_cols = []

    if p > 0:
        X_cols += [f"{y_col}_lag{i}" for i in range(1, p + 1)]

    X_cols += [f"{x_col}_lag{i}" for i in range(0, q + 1)]

    X = sm.add_constant(temp[X_cols])

    model = sm.OLS(y, X).fit()

    rmse = np.sqrt((model.resid ** 2).mean())

    return {
        "p": p,
        "q": q,
        "AIC": model.aic,
        "BIC": model.bic,
        "RMSE": rmse
    }

In [12]:
results_us = []

for p in range(0, 6):
    for q in range(0, 6):
        try:
            res = run_adl(df, "d_gas_us", "d_wti", p, q)
            results_us.append(res)
        except Exception as e:
            print(f"Error at p={p}, q={q}: {e}")

results_us_df = pd.DataFrame(results_us).sort_values("BIC")
print(results_us_df.head(25))

    p  q         AIC         BIC      RMSE
7   1  1 -821.188364 -806.945638  0.049118
13  2  1 -818.048382 -800.264242  0.048924
8   1  2 -816.777802 -798.993662  0.049044
14  2  2 -817.611032 -796.270063  0.048777
19  3  1 -815.728700 -794.410943  0.048652
9   1  3 -812.784519 -791.466762  0.048930
20  3  2 -814.501536 -789.630819  0.048579
15  2  3 -812.289806 -787.419089  0.048788
25  4  1 -810.239146 -785.395614  0.048679
21  3  3 -812.660347 -784.236670  0.048564
10  1  4 -806.900515 -782.056982  0.048996
26  4  2 -808.860806 -780.468197  0.048620
16  2  4 -806.628879 -778.236270  0.048831
31  5  1 -805.042883 -776.681464  0.048678
27  4  3 -807.149548 -775.207864  0.048592
22  3  4 -806.987473 -775.045788  0.048608
11  1  5 -801.776363 -773.414943  0.048989
32  5  2 -803.595394 -771.688797  0.048625
28  4  4 -805.314558 -769.823797  0.048577
17  2  5 -801.418313 -769.511716  0.048832
3   0  3 -785.139901 -767.375103  0.051824
23  3  5 -802.474039 -767.022265  0.048542
33  5  3 -8

In [8]:
results_us = []

for p in range(0, 6):
    for q in range(0, 6):
        try:
            res = run_adl(df, "d_gas_austin", "d_wti", p, q)
            results_us.append(res)
        except Exception as e:
            print(f"Error at p={p}, q={q}: {e}")

results_us_df = pd.DataFrame(results_us).sort_values("BIC")
print(results_us_df.head(25))

    p  q         AIC         BIC      RMSE
14  2  2 -453.544810 -432.203842  0.098502
8   1  2 -448.006752 -430.222612  0.099946
9   1  3 -447.227654 -425.909897  0.099369
15  2  3 -449.267990 -424.397273  0.098594
20  3  2 -449.045673 -424.174956  0.098637
10  1  4 -444.598087 -419.754554  0.099147
16  2  4 -447.423337 -419.030728  0.098220
21  3  3 -447.268443 -418.844766  0.098594
26  4  2 -445.907287 -417.514678  0.098511
22  3  4 -446.037970 -414.096285  0.098103
27  4  3 -444.131116 -412.189432  0.098468
11  1  5 -439.834555 -411.473136  0.099337
17  2  5 -442.633251 -410.726654  0.098410
32  5  2 -441.352732 -409.446135  0.098656
28  4  4 -444.038039 -408.547278  0.098103
13  2  1 -423.656660 -405.872520  0.104757
23  3  5 -441.238949 -405.787175  0.098294
33  5  3 -439.593140 -404.141366  0.098610
34  5  4 -439.592530 -400.595578  0.098226
29  4  5 -439.241378 -400.244426  0.098293
19  3  1 -419.190600 -397.872842  0.104918
35  5  5 -437.600089 -395.057960  0.098224
7   1  1 -4